In [0]:
base_path = "/Volumes/capstone_project/capstone_schema/capstone_volume"

In [0]:
import dlt
from pyspark.sql.functions import *

# Define the base path where your 2M rows of messy JSON are stored
# In a real setup, this would be a variable or a config setting 

@dlt.table(
    name="bronze_orders",
    comment="Raw e-commerce order events ingested via Auto Loader",
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.zOrderCols": "order_id" # Optimization for later joins
    },
    partition_cols=["event_date"] # Partitioning by date as per requirements
)
def bronze_orders():
    return (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            # 1. Schema Inference: Let Spark figure out the columns from the JSON
            .option("cloudFiles.inferColumnTypes", "true")
            # 2. Rescued Data: Automatically catch malformed rows (broken JSON/wrong types)
            .option("cloudFiles.schemaEvolutionMode", "rescue")
            .load(base_path)
            # Add a processing timestamp and a date column for partitioning
            .withColumn("processing_time", current_timestamp())
            .withColumn("event_date", col("event_timestamp").cast("date"))
    )